# 02 — Retornos y distribución empírica: preparación de entradas

Esta notebook prepara y valida las entradas locales del corte final antes de cualquier transformación o análisis estadístico. Carga la vista elegible de precios ajustados Gold y la matriz de retornos simples correspondiente. No descarga datos, no usa credenciales y no implementa todavía distribución empírica, diagnósticos, escenarios ni carteras.

## Alcance actual

El corte aprobado cubre 2018-01-02 a 2026-10-09: 2,205 fechas de precios y 2,204 retornos diarios utilizables por instrumento. La matriz Gold de precios conserva 138 series para auditoría; aquí se selecciona explícitamente, y sin persistirla, la vista de las 125 columnas que pertenecen al universo elegible.

La primera fila de retornos se conserva durante la validación porque representa la ausencia prevista de un precio anterior aprobado. Sólo después se crea una vista de retornos utilizables retirando esa fila conocida; no se usa `drop_nulls` ni se imputan datos.

In [1]:
from __future__ import annotations

import hashlib
import json
import site
import sys
from datetime import date
from pathlib import Path

def find_project_root() -> Path:
    def is_project_root(candidate: Path) -> bool:
        return ((candidate / 'config' / 'universe.csv').is_file()
                and (candidate / 'src' / 'scenario_portfolio_lab').is_dir())

    for origin in (Path.cwd(), *Path.cwd().parents):
        for candidate in (origin, origin / 'scenario-portfolio-lab', origin / 'data-science-portfolio' / 'scenario-portfolio-lab'):
            if is_project_root(candidate):
                return candidate
    raise RuntimeError(
        'Repository files config/universe.csv and src/scenario_portfolio_lab were not found. '
        'Open the notebook from the complete scenario-portfolio-lab repository (or its data-science-portfolio parent). '
        'Do not run it from an exported standalone .ipynb file.'
    )


ROOT = find_project_root()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# MyST/Jupyter kernels may differ from the project virtual environment.
# Reuse already installed local dependencies when this Windows project .venv exists; never install packages here.
LOCAL_SITE_PACKAGES = ROOT / '.venv' / 'Lib' / 'site-packages'
if LOCAL_SITE_PACKAGES.is_dir():
    site.addsitedir(str(LOCAL_SITE_PACKAGES))

try:
    import exchange_calendars as xcals
    import polars as pl
except ModuleNotFoundError as exc:
    raise RuntimeError(
        'Missing project dependencies. Select the project .venv Python 3.13.2 kernel or install requirements.txt '
        'into the active kernel environment before running this notebook.'
    ) from exc

RUN_ROOT = ROOT / 'data' / 'published' / 'cut_2026-10-09_run_2026-10-11_full_window_final'
GOLD_ROOT = RUN_ROOT / 'gold'
PRICE_MATRIX_PATH = GOLD_ROOT / 'adjusted_close_matrix.parquet'
RETURN_MATRIX_PATH = GOLD_ROOT / 'eligible_simple_total_return_matrix.parquet'
RUN_MANIFEST_PATH = RUN_ROOT / 'run_manifest.json'
ELIGIBLE_CATALOG_PATH = RUN_ROOT / 'reports' / 'eligible_universe.csv'

for required in (PRICE_MATRIX_PATH, RETURN_MATRIX_PATH, RUN_MANIFEST_PATH, ELIGIBLE_CATALOG_PATH):
    if not required.is_file():
        raise FileNotFoundError(f'Required final-cut artifact is missing: {required.relative_to(ROOT)}')


## Carga del manifiesto y catálogo elegible

El manifiesto final contiene hashes de los 138 raw que alimentaron el corte. Esta notebook los comprueba contra sus referencias previas. El manifiesto no contiene hashes de referencia para los dos Parquet matriciales; por ello valida su estructura y contenido, pero no los presenta como verificados criptográficamente contra una referencia previa.

In [2]:
run_manifest = json.loads(RUN_MANIFEST_PATH.read_text(encoding='utf-8'))
approved_period = run_manifest['approved_period']
assert approved_period == {'start': '2018-01-02', 'end': '2026-10-09'}

eligible_catalog = pl.read_csv(ELIGIBLE_CATALOG_PATH).filter(pl.col('universe_status') == 'ELIGIBLE')
eligible_symbols = eligible_catalog['symbol'].to_list()
assert len(eligible_symbols) == 125 and len(set(eligible_symbols)) == 125

raw_hash_checks = []
for symbol, reference in run_manifest['raw'].items():
    raw_path = ROOT / Path(reference['path'].replace('\\', '/'))
    if not raw_path.is_file():
        raise FileNotFoundError(f'Manifest-referenced raw is missing for {symbol}.')
    raw_hash_checks.append(hashlib.sha256(raw_path.read_bytes()).hexdigest() == reference['sha256'])
assert len(raw_hash_checks) == 138 and all(raw_hash_checks)


## Carga y validación de matrices

Las sesiones esperadas se reconstruyen desde el calendario común documentado en el manifiesto (`exchange_calendars:US_EQUITY_COMMON(XNYS=XNAS=ARCX)`). Las validaciones son explícitas: fechas, orden de símbolos, precios, nulos previstos en retornos y finitud.

In [3]:
all_gold_prices = pl.read_parquet(PRICE_MATRIX_PATH)
return_matrix = pl.read_parquet(RETURN_MATRIX_PATH)

assert all_gold_prices.columns[0] == return_matrix.columns[0] == 'session_date'
return_symbols = return_matrix.columns[1:]
assert return_symbols == eligible_symbols
assert set(return_symbols).issubset(set(all_gold_prices.columns[1:]))

# Explicit ordered view: not a new persisted matrix and not a silent filter.
eligible_prices = all_gold_prices.select(['session_date', *return_symbols])
assert eligible_prices.width == return_matrix.width == 126
assert eligible_prices.height == return_matrix.height == 2205

start, end = date.fromisoformat(approved_period['start']), date.fromisoformat(approved_period['end'])
expected_dates = [stamp.date() for stamp in xcals.get_calendar('XNYS').sessions if start <= stamp.date() <= end]
for matrix in (eligible_prices, return_matrix):
    dates = matrix['session_date'].to_list()
    assert dates == expected_dates
    assert dates == sorted(dates) and len(set(dates)) == len(dates) == 2205

price_values = eligible_prices.select(pl.exclude('session_date'))
assert all(value == 0 for value in price_values.null_count().row(0))
assert not any(price_values.select(pl.all().is_nan().any()).row(0))
assert not any(price_values.select(pl.all().is_infinite().any()).row(0))
assert not any(price_values.select((pl.all() <= 0).any()).row(0))

first_return_row = return_matrix.slice(0, 1).select(pl.exclude('session_date'))
subsequent_return_rows = return_matrix.slice(1).select(pl.exclude('session_date'))
assert all(value == 1 for value in return_matrix.select(pl.exclude('session_date').null_count()).row(0))
assert all(value == 1 for value in first_return_row.null_count().row(0))
assert all(value == 0 for value in subsequent_return_rows.null_count().row(0))
assert not any(subsequent_return_rows.select(pl.all().is_nan().any()).row(0))
assert not any(subsequent_return_rows.select(pl.all().is_infinite().any()).row(0))


## Vista explícita de retornos utilizables

La siguiente vista retira solamente la primera fila prevista, ya comprobada como nula para los 125 instrumentos. Conserva las fechas y el orden explícito de símbolos de la matriz elegible. No se calculan transformaciones ni estadísticas.

In [4]:
usable_returns = return_matrix.slice(1).select(['session_date', *return_symbols])
assert usable_returns.shape == (2204, 126)
assert usable_returns['session_date'].to_list() == expected_dates[1:]
assert usable_returns.columns[1:] == return_symbols
assert all(value == 0 for value in usable_returns.select(pl.exclude('session_date').null_count()).row(0))

validation_summary = pl.DataFrame({
    'check': [
        'Approved period',
        'Expected and observed price dates',
        'Eligible symbols',
        'Eligible adjusted-price view',
        'Eligible simple-return matrix',
        'Usable-return view',
        'Manifest-referenced raw hashes',
        'Matrix reference hashes',
    ],
    'result': [
        '2018-01-02 to 2026-10-09',
        '2,205 ordered, unique XNYS common sessions',
        '125 exact symbols, explicit order',
        '2,205 rows × (date + 125); positive, finite, no nulls',
        '2,205 rows × (date + 125); first-row null only, finite thereafter',
        '2,204 rows × (date + 125); dates retained, no nulls',
        '138/138 verified against manifest references',
        'Not available in run_manifest; no new hash is treated as prior evidence',
    ],
})
validation_summary


check,result
str,str
"""Approved period""","""2018-01-02 to 2026-10-09"""
"""Expected and observed price da…","""2,205 ordered, unique XNYS com…"
"""Eligible symbols""","""125 exact symbols, explicit or…"
"""Eligible adjusted-price view""","""2,205 rows × (date + 125); pos…"
"""Eligible simple-return matrix""","""2,205 rows × (date + 125); fir…"
"""Usable-return view""","""2,204 rows × (date + 125); dat…"
"""Manifest-referenced raw hashes""","""138/138 verified against manif…"
"""Matrix reference hashes""","""Not available in run_manifest;…"


# 1. Retornos simples y log-retornos

Con los precios ajustados, el retorno simple es $r_t=P_t^{adj}/P_{t-1}^{adj}-1$. El log-retorno es $\ell_t=\log(1+r_t)$ y permite recuperar el retorno simple con $r_t=\exp(\ell_t)-1$.

Los cálculos usan decimales: `0.01` se presenta como `1%`. La transformación cambia la representación, no el rendimiento económico; no filtra, recorta ni imputa observaciones.

In [5]:
import numpy as np

simple_return_values = usable_returns.select(pl.exclude('session_date')).to_numpy()
domain_failures = np.argwhere(simple_return_values <= -1.0)
if domain_failures.size:
    examples = [{'session_date': str(usable_returns['session_date'][int(i)]), 'symbol': return_symbols[int(j)], 'simple_return': float(simple_return_values[i, j])} for i, j in domain_failures[:10]]
    raise ValueError(f'Log-return domain failure (r <= -1); first locations: {examples}')
log_return_values = np.log1p(simple_return_values)
if not np.isfinite(log_return_values).all():
    raise ValueError('Log-return transformation produced a non-finite value.')
log_returns = pl.DataFrame({'session_date': usable_returns['session_date'], **{symbol: log_return_values[:, i] for i, symbol in enumerate(return_symbols)}})
assert log_returns.columns == usable_returns.columns and log_returns.shape == (2204, 126)
assert all(value == 0 for value in log_returns.select(pl.exclude('session_date').null_count()).row(0))

# Rule: first usable date and first three symbols in the explicit matrix order; no performance-based selection.
illustration_rows = []
for symbol in return_symbols[:3]:
    log_return = float(log_returns[symbol][0])
    illustration_rows.append({'session_date': usable_returns['session_date'][0], 'symbol': symbol, 'adjusted_price_previous': float(eligible_prices[symbol][0]), 'adjusted_price_current': float(eligible_prices[symbol][1]), 'simple_return_decimal': float(usable_returns[symbol][0]), 'log_return': log_return, 'recovered_simple_return': float(np.expm1(log_return))})
illustration_table = pl.DataFrame(illustration_rows)
illustration_table


session_date,symbol,adjusted_price_previous,adjusted_price_current,simple_return_decimal,log_return,recovered_simple_return
date,str,f64,f64,f64,f64,f64
2018-01-03,"""AA""",52.459843,51.822756,-0.012144,-0.012219,-0.012144
2018-01-03,"""AAL""",51.648665,51.015119,-0.012266,-0.012342,-0.012266
2018-01-03,"""AAPL""",40.233642,40.226635,-0.000174,-0.000174,-0.000174


## Comprobaciones sobre los 125 instrumentos

Las identidades se aplican a las 2,204 observaciones de los 125 instrumentos completos. Las tolerancias de doble precisión son absoluta `1e-12` y relativa `1e-10`; el error relativo se reporta sólo si la referencia no es cero y, si lo es, controla la tolerancia absoluta. No se redondea antes de validar.

In [6]:
ABSOLUTE_TOLERANCE, RELATIVE_TOLERANCE = 1e-12, 1e-10
def identity_metrics(actual, reference):
    absolute_error = np.abs(actual - reference)
    nonzero = np.abs(reference) > 0.0
    relative_error = np.full(reference.shape, np.nan); relative_error[nonzero] = absolute_error[nonzero] / np.abs(reference[nonzero])
    return {'max_absolute_error': float(absolute_error.max()), 'max_relative_error_nonzero_reference': float(np.nanmax(relative_error)) if nonzero.any() else None, 'zero_reference_count': int((~nonzero).sum()), 'passed': bool(np.all(absolute_error <= ABSOLUTE_TOLERANCE + RELATIVE_TOLERANCE * np.abs(reference)))}

round_trip = identity_metrics(np.expm1(log_return_values), simple_return_values)
previous_prices = eligible_prices.slice(0, 2204).select(pl.exclude('session_date')).to_numpy()
current_prices = eligible_prices.slice(1, 2204).select(pl.exclude('session_date')).to_numpy()
assert usable_returns['session_date'].to_list() == eligible_prices['session_date'].slice(1, 2204).to_list()
assert usable_returns.columns[1:] == eligible_prices.columns[1:] == log_returns.columns[1:]
price_correspondence = identity_metrics(log_return_values, np.log(current_prices) - np.log(previous_prices))
simple_compounded = np.prod(1.0 + simple_return_values, axis=0) - 1.0
log_compounded = np.expm1(np.sum(log_return_values, axis=0))
endpoint_compounded = current_prices[-1] / previous_prices[0] - 1.0
checks = [('Round trip: expm1(log1p(r)) vs r', round_trip), ('Price correspondence: log1p(r) vs log(P_t)-log(P_t-1)', price_correspondence), ('Time composition: simple product vs log sum', identity_metrics(simple_compounded, log_compounded)), ('Time composition: simple product vs price endpoints', identity_metrics(simple_compounded, endpoint_compounded)), ('Time composition: log sum vs price endpoints', identity_metrics(log_compounded, endpoint_compounded))]
failures = [(name, metric) for name, metric in checks if not metric['passed']]
if failures: raise AssertionError(f'Identity validation failed without altering data: {failures}')
identity_summary = pl.DataFrame({'check': [name for name, _ in checks], 'max_absolute_error': [metric['max_absolute_error'] for _, metric in checks], 'max_relative_error_nonzero_reference': [metric['max_relative_error_nonzero_reference'] for _, metric in checks], 'zero_reference_count': [metric['zero_reference_count'] for _, metric in checks], 'result': ['PASS'] * len(checks)})
run_summary = pl.DataFrame({'item': ['Effective return period', 'Instruments', 'Usable observations per instrument', 'Log-return dimensions', 'Absolute tolerance', 'Relative tolerance'], 'value': [f"{usable_returns['session_date'][0]} to {usable_returns['session_date'][-1]}", '125', '2,204', '2,204 rows × (date + 125 instruments)', f'{ABSOLUTE_TOLERANCE:.0e}', f'{RELATIVE_TOLERANCE:.0e}']})
run_summary, identity_summary


(shape: (6, 2)
 ┌─────────────────────────────────┬─────────────────────────────────┐
 │ item                            ┆ value                           │
 │ ---                             ┆ ---                             │
 │ str                             ┆ str                             │
 ╞═════════════════════════════════╪═════════════════════════════════╡
 │ Effective return period         ┆ 2018-01-03 to 2026-10-09        │
 │ Instruments                     ┆ 125                             │
 │ Usable observations per instru… ┆ 2,204                           │
 │ Log-return dimensions           ┆ 2,204 rows × (date + 125 instr… │
 │ Absolute tolerance              ┆ 1e-12                           │
 │ Relative tolerance              ┆ 1e-10                           │
 └─────────────────────────────────┴─────────────────────────────────┘,
 shape: (5, 5)
 ┌──────────────────────┬────────────────────┬──────────────────────┬──────────────────────┬────────┐
 │ check       

## Interpretación

Los retornos simples se componen multiplicativamente; para un mismo instrumento, los log-retornos se suman a través del tiempo. El retorno simple de una cartera durante una sesión se calcula ponderando retornos simples, no ponderando directamente log-retornos.

Obtener log-retornos no demuestra estacionariedad. Los acumulados sobre `adjusted_close` siguen la convención del proveedor y no representan automáticamente una cuenta operada con costes e impuestos.

## Siguiente paso: transformaciones de retornos

Pendiente de desarrollar con el usuario. Esta notebook se detiene después de la carga y validación de entradas.